# TUGAS MANDIRI PRAKTIKUM BIG DATA
## Pertemuan 10 — Pemrosesan Data Streaming dengan Spark Structured Streaming

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |
| **Nama** | Sahrul Indra Safani |
| **NPM** | 2505060042 |
| **Studi Kasus** | Kopi Nusantara |

## Inisialisasi SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
from datetime import datetime
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10_2505060042_Sahrul_Indra_Safani") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Menghentikan streaming query sebelumnya jika ada yang masih aktif
for q in spark.streams.active:
    q.stop()

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


## Bagian A. Menyiapkan Streaming Source

In [2]:
# Mendefinisikan skema data pesanan Kopi Nusantara
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

folder_stream = "tugas10_stream"
shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Streaming DataFrame siap.")
print("df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame siap.
df_stream.isStreaming: True


## Bagian B. Agregasi per Cabang

In [3]:
for q in spark.streams.active:
    if q.name == "agregasi_cabang":
        q.stop()

# Menambahkan kolom total_harga
df_pesanan = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per nama_cabang
agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query 1 dimulai. Status:", query1.status)

Streaming query 1 dimulai. Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


## Bagian C. Agregasi Berbasis Window Waktu

In [4]:
for q in spark.streams.active:
    if q.name == "agregasi_waktu":
        q.stop()

df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

df_stream_waktu = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi window 2 menit
agregasi_window = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan_window"),
    count("pesanan_id").alias("jumlah_pesanan_window")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query 2 dimulai. Status:", query2.status)

Streaming query 2 dimulai. Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Bagian D. Menguji dengan Data Simulasi

In [5]:
# Batch 1: Malioboro, Kaliurang, Kotagede
waktu_sekarang = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003", "P004"],
    "nama_cabang": ["Malioboro", "Kaliurang", "Kotagede", "Malioboro"],
    "nama_menu": ["Kopi Susu Gula Aren", "Americano", "Cafe Latte", "Croissant"],
    "qty": [2, 1, 2, 3],
    "harga_satuan": [18000, 15000, 22000, 20000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 4
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(8)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan_window, jumlah_pesanan_window FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

Batch 1 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|  Malioboro|           96000|             2|
|   Kotagede|           44000|             1|
|  Kaliurang|           15000|             1|
+-----------+----------------+--------------+


+-------------------+-------------------+-----------------------+---------------------+
|start              |end                |total_pendapatan_window|jumlah_pesanan_window|
+-------------------+-------------------+-----------------------+---------------------+
|2026-11-11 21:50:00|2026-11-11 21:52:00|155000                 |4                    |
+-------------------+-------------------+-----------------------+---------------------+



In [6]:
# Batch 2: Malioboro, Kotagede, Solo
waktu_sekarang = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P005", "P006", "P007", "P008"],
    "nama_cabang": ["Malioboro", "Kotagede", "Solo", "Solo"],
    "nama_menu": ["V60 Filter", "Roti Bakar", "Espresso", "Kopi Susu Gula Aren"],
    "qty": [1, 2, 2, 1],
    "harga_satuan": [28000, 18000, 16000, 18000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 4
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(8)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan_window, jumlah_pesanan_window FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

Batch 2 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|  Malioboro|          124000|             3|
|   Kotagede|           80000|             2|
|       Solo|           50000|             2|
|  Kaliurang|           15000|             1|
+-----------+----------------+--------------+


+-------------------+-------------------+-----------------------+---------------------+
|start              |end                |total_pendapatan_window|jumlah_pesanan_window|
+-------------------+-------------------+-----------------------+---------------------+
|2026-11-11 21:50:00|2026-11-11 21:52:00|269000                 |8                    |
+-------------------+-------------------+-----------------------+---------------------+



In [7]:
# Batch 3: Kaliurang, Solo, Magelang
waktu_sekarang = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P009", "P010", "P011", "P012"],
    "nama_cabang": ["Kaliurang", "Solo", "Magelang", "Magelang"],
    "nama_menu": ["Mocha", "Cappuccino", "Caramel Macchiato", "French Fries"],
    "qty": [2, 1, 2, 2],
    "harga_satuan": [26000, 24000, 27000, 16000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 4
}).to_csv(f"{folder_stream}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(8)

spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan_window, jumlah_pesanan_window FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

Batch 3 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|  Malioboro|          124000|             3|
|   Magelang|           86000|             2|
|   Kotagede|           80000|             2|
|       Solo|           74000|             3|
|  Kaliurang|           67000|             2|
+-----------+----------------+--------------+


+-------------------+-------------------+-----------------------+---------------------+
|start              |end                |total_pendapatan_window|jumlah_pesanan_window|
+-------------------+-------------------+-----------------------+---------------------+
|2026-11-11 21:50:00|2026-11-11 21:52:00|431000                 |12                   |
+-------------------+-------------------+-----------------------+---------------------+



## Bagian E. Sink Berbasis Berkas (Tantangan)

In [8]:
# Menyiapkan folder output dan checkpoint
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

print("Streaming query 3 dimulai. Status:", query3.status)

Streaming query 3 dimulai. Status: {'message': 'Initializing StreamExecution', 'isDataAvailable': False, 'isTriggerActive': False}


In [9]:
# Mengirim batch data baru untuk menguji penulisan ke Parquet
waktu_sekarang = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P013", "P014", "P015"],
    "nama_cabang": ["Malioboro", "Magelang", "Kotagede"],
    "nama_menu": ["Matcha Latte", "Espresso", "Croissant"],
    "qty": [2, 1, 3],
    "harga_satuan": [24000, 16000, 20000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder_stream}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark memproses...")
time.sleep(10)

# Memeriksa berkas Parquet yang tersimpan
print("Daftar berkas di tugas10_output:")
for berkas in sorted(os.listdir(folder_output)):
    if not berkas.startswith("."):
        print("-", berkas)

# Membaca kembali data Parquet dengan DataFrame batch
df_parquet = spark.read.parquet(folder_output)
print("Jumlah baris data tersimpan:", df_parquet.count())
df_parquet.show(truncate=False)

Batch 4 dikirim. Menunggu Spark memproses...
Daftar berkas di tugas10_output:
- _spark_metadata
- part-00000-98d002c5-397d-45b9-8b7f-907ff08b3982-c000.snappy.parquet
- part-00001-db60b003-ba2b-4874-a3e7-dbc78a1c6b3b-c000.snappy.parquet
- part-00002-376b3f2a-fc95-4a70-bf1e-800cf9499ed7-c000.snappy.parquet
- part-00003-eb6c3fb5-3c80-4f9b-b08b-a02a92013eb9-c000.snappy.parquet
Jumlah baris data tersimpan: 15
+----------+-----------+-------------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu          |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-------------------+---+------------+-------------------+
|P001      |Malioboro  |Kopi Susu Gula Aren|2  |18000       |2026-11-11 21:51:14|
|P002      |Kaliurang  |Americano          |1  |15000       |2026-11-11 21:51:14|
|P003      |Kotagede   |Cafe Latte         |2  |22000       |2026-11-11 21:51:14|
|P004      |Malioboro  |Croissant          |3  |20000       |2026-11-11 21:51:14|
|P005      |Maliob

## Penutup

In [10]:
# Menghentikan seluruh streaming query dan menutup SparkSession
query1.stop()
query2.stop()
query3.stop()
spark.stop()

print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.
